In [0]:
from delta.tables import *
from pyspark.sql import DataFrame

def create_table(snapshot_date:str):
  if snapshot_date == "1900-01-01":

    spark.sql("create catalog if not exists patterns")
    spark.sql("create schema if not exists patterns.dw")
    spark.sql("drop table if exists patterns.dw.type2")

    spark.sql("""
      create table patterns.dw.type2 (
        quid long not null,
        firstname string,
        surname string,
        _created_by_job_run_id long not null,
        _created_date timestamp not null,
        _created_by string not null,
        _updated_by_job_run_id long,
        _updated_date timestamp,
        _updated_by string not null,
        _version int not null,
        _from_date date not null,
        _to_date date not null,
        _is_current boolean not null,
        _cdc_operation string not null,
        _change_hash string not null
      )""")
    
def get_source_data(id:int, snapshot_date:str, firstname:str, surname:str):
  df = spark.sql(f"""
    SELECT
      cast({str(id)} as long) as quid, 
      "{firstname}" as firstname, 
      "{surname}" as surname,
      cast({job_run_id} as long) as _created_by_job_run_id,
      current_date() as _created_date,
      current_user() as _created_by,
      cast({job_run_id} as long) as _updated_by_job_run_id,
      current_date() as _updated_date,
      current_user() as _updated_by,
      cast(1 as int) as _version,
      to_date('{snapshot_date}') as _from_date,
      date_sub(cast("9999-12-31" as date), 1) as _to_date,
      true as _is_current,
      "I" as _cdc_operation,
      md5(concat_ws("|",firstname, surname)) as _change_hash
  """)
  return df

def merge_inserts(df:DataFrame, job_run_id:int, snapshot_date:str):

  destination = DeltaTable.forName(spark, "patterns.dw.type2")

  audit = (
    destination.alias('dst') 
    .merge(
      df.alias('src'),
      """src.quid = dst.quid 
      and dst._is_current = true"""
    )
    .whenNotMatchedInsert(
      values={
        "quid" : "src.quid",
        "firstname" : "src.firstname",
        "surname" : "src.surname",
        "_created_by_job_run_id" : "src._created_by_job_run_id",
        "_created_date" : "src._created_date",
        "_created_by" : "src._created_by",
        "_updated_by_job_run_id" : "src._updated_by_job_run_id",
        "_updated_date" : "src._updated_date",
        "_updated_by" : "src._updated_by",
        "_version" : "src._version",
        "_from_date" : "src._from_date",
        "_to_date" : "src._to_date",
        "_is_current" : "src._is_current",
        "_cdc_operation" : "src._cdc_operation",
        "_change_hash" : "src._change_hash",
      }
    )
    .whenMatchedUpdate(
      condition="dst._change_hash != src._change_hash or dst._cdc_operation = 'D'",
      set =
      {
        "_updated_by_job_run_id" : "src._updated_by_job_run_id",
        "_updated_date" : "src._updated_date",
        "_updated_by" : "src._updated_by",
        "_to_date" : "src._from_date",
        "_is_current" : "false",
      }
    )
    .whenNotMatchedBySourceUpdate(
      condition="dst._is_current=true",
      set = {
        "_updated_by_job_run_id" : str(job_run_id),
        "_updated_date" : "current_timestamp()",
        "_updated_by" : "current_user()",
        "_to_date" : f"cast('{snapshot_date}' as date)",
        "_cdc_operation" : "'D'"
      }
    )
    .execute()
  )
  return audit

def get_insert_updates(df:DataFrame):
  df_updates = spark.sql("""
    select src.* except(_version, _cdc_operation),
    dst._version+1 as _version,
    if (dst._cdc_operation='D', 'I', 'U') as _cdc_operation
    from patterns.dw.type2 dst
    join {df} src 
      on src.quid = dst.quid 
      and src._updated_by_job_run_id = dst._updated_by_job_run_id
    where 
      dst._is_current=false
      and (
        -- updates
        (dst._cdc_operation not in ('D')
        and dst._change_hash != src._change_hash)
        -- deletes
        or dst._cdc_operation = 'D'
      )
  """, df=df)
  return df_updates


def insert_updates(df:DataFrame):
  destination = DeltaTable.forName(spark, "patterns.dw.type2")

  audit = (
    destination.alias('dst') \
    .merge(
      df.alias('src'),
      'src.quid = dst.quid and dst._is_current = true'
    )
    .whenNotMatchedInsert(
      values={
        "quid" : "src.quid",
        "firstname" : "src.firstname",
        "surname" : "src.surname",
        "_created_by_job_run_id" : "src._created_by_job_run_id",
        "_created_date" : "src._created_date",
        "_created_by" : "src._created_by",
        "_updated_by_job_run_id" : "src._updated_by_job_run_id",
        "_updated_date" : "src._updated_date",
        "_updated_by" : "src._updated_by",
        "_version" : "src._version",
        "_from_date" : "src._from_date",
        "_to_date" : "src._to_date",
        "_is_current" : "src._is_current",
        "_cdc_operation" : "src._cdc_operation",
        "_change_hash" : "src._change_hash",
      }
    )
    .execute()
  )
  return audit

In [0]:
%python
data = [
  {"id":1, "firstname":"Walter", "surname": "white1", "snapshot_date":"1900-01-01", "job_run_id":19000101},
  {"id":1, "firstname":"Walter", "surname": "white2", "snapshot_date":"1900-01-02", "job_run_id":19000102},
  {"id":1, "firstname":"Walter", "surname": "white3", "snapshot_date":"1900-01-03", "job_run_id":19000103},
  {"id":1, "firstname":"Walter", "surname": "white4", "snapshot_date":"1900-01-04", "job_run_id":19000104},
  {"id":1, "firstname":"Walter", "surname": "white5", "snapshot_date":"1900-01-05", "job_run_id":19000105},
  {"id":2, "firstname":"Truman", "surname": "Burbank1", "snapshot_date":"1900-01-06", "job_run_id":19000106},
  {"id":2, "firstname":"Truman", "surname": "Burbank2", "snapshot_date":"1900-01-07", "job_run_id":19000107},
  {"id":1, "firstname":"Walter", "surname": "white5", "snapshot_date":"1900-01-08", "job_run_id":19000108},
  {"id":1, "firstname":"Walter", "surname": "white6", "snapshot_date":"1900-01-09", "job_run_id":19000109},
  {"id":1, "firstname":"Walter", "surname": "white7", "snapshot_date":"1900-01-10", "job_run_id":19000110},
  {"id":1, "firstname":"Walter", "surname": "white8", "snapshot_date":"1900-01-11", "job_run_id":19000111},
  {"id":1, "firstname":"Walter", "surname": "white9", "snapshot_date":"1900-01-12", "job_run_id":19000112},
  {"id":2, "firstname":"Truman", "surname": "Burbank3", "snapshot_date":"1900-01-13", "job_run_id":19000113},
  {"id":2, "firstname":"Truman", "surname": "Burbank4", "snapshot_date":"1900-01-14", "job_run_id":19000114},
  {"id":1, "firstname":"Walter", "surname": "white1", "snapshot_date":"1900-01-15", "job_run_id":19000115},
  {"id":1, "firstname":"Walter", "surname": "white2", "snapshot_date":"1900-01-16", "job_run_id":19000116},
  {"id":1, "firstname":"Walter", "surname": "white3", "snapshot_date":"1900-01-17", "job_run_id":19000117},
]

for d in data:
  id = d["id"]
  snapshot_date = d["snapshot_date"]
  surname = d["surname"]
  firstname = d["firstname"]
  job_run_id = d["job_run_id"]


  create_table(snapshot_date)
  df = get_source_data(id, snapshot_date, firstname, surname)
  audit = merge_inserts(df, job_run_id, snapshot_date)
  df_insert_updates = get_insert_updates(df)
  audit = insert_updates(df_insert_updates)




In [0]:
%sql

select * from patterns.dw.type2 where quid = 1 order by _from_date, `_version`